# CNN Encoder-Decoder
This code was originally purpoused for turning images took in the day to nightime images, but the dataset was unpaired and I then had to repurpouse. It now uses the edge to shoes dataset (Edges2Shoes on kaggle). This is a simple CNN that turns images to images using an encoder-decoder architecture. The training code is below.

In [1]:
# Training code

import tensorflow as tf
import cv2
import os
from tqdm import tqdm
import shutil

def split_imgs(input_folder, output_A, output_B):
    if os.path.exists(output_A):
        shutil.rmtree(output_A)
    if os.path.exists(output_B):
        shutil.rmtree(output_B)
    
    os.makedirs(output_A, exist_ok=True)
    os.makedirs(output_B, exist_ok=True)
    
    for filename in tqdm(os.listdir(input_folder)):
        path = os.path.join(input_folder, filename)
        img = cv2.imread(path)
    
        h, w, _ = img.shape
        midpoint = w // 2
    
        left = img[:, :midpoint]
        right = img[:, midpoint:]
    
        cv2.imwrite(os.path.join(output_A, filename), left)
        cv2.imwrite(os.path.join(output_B, filename), right)

tf.keras.mixed_precision.set_global_policy('mixed_float16')

train_path = "/kaggle/input/datasets/balraj98/edges2shoes-dataset/train"
train_X = "/kaggle/working/train_X"
train_y = "/kaggle/working/train_y"

val_path = "/kaggle/input/datasets/balraj98/edges2shoes-dataset/val"
val_X = "/kaggle/working/val_X"
val_y = "/kaggle/working/val_y"

split_imgs(val_path, val_X, val_y)
split_imgs(train_path, train_X, train_y)

print(len(os.listdir('/kaggle/working/train_X')))
print(len(os.listdir('/kaggle/working/train_y')))

img_height, img_width = 256, 256
batch_size = 8

X = tf.keras.utils.image_dataset_from_directory(
  train_X,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

y = tf.keras.utils.image_dataset_from_directory(
  train_y,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

X_val = tf.keras.utils.image_dataset_from_directory(
  val_X,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

y_val = tf.keras.utils.image_dataset_from_directory(
  val_y,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  label_mode=None,
  shuffle=False)

X_val = X_val.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)
y_val = y_val.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)

X = X.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)
y = y.map(lambda x: x / 255.0).prefetch(tf.data.AUTOTUNE)

inputs = tf.keras.layers.Input(shape=(256,256,3))

c1 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
c2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c1)

p1 = tf.keras.layers.MaxPooling2D()(c2)

b1 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
b2 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(b1)

u1 = tf.keras.layers.UpSampling2D()(b2)

u2 = tf.keras.layers.Concatenate()([u1, c2])

c3 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(u2)
c4 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c3)

outputs = tf.keras.layers.Conv2D(3,1, activation='sigmoid', padding='same', dtype='float32')(c2)

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam', loss='mae')

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

dataset = tf.data.Dataset.zip((X, y))
val_dataset = tf.data.Dataset.zip((X_val, y_val))

model.fit(dataset, epochs=20, callbacks=[early_stopping], validation_data=val_dataset)
model.save('/kaggle/working/edges-to-shoes.keras')

2026-05-26 11:46:03.288670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779795963.474602      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779795963.525613      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779795963.914283      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779795963.914325      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779795963.914328      23 computation_placer.cc:177] computation placer alr

49825
49825
Found 49825 files.


I0000 00:00:1779796506.882990      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 49825 files.
Found 200 files.
Found 200 files.
Epoch 1/20


I0000 00:00:1779796510.299616      76 service.cc:152] XLA service 0x7bee2c08ae80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779796510.299657      76 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1779796510.678479      76 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779796520.208617      76 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6229/6229 ━━━━━━━━━━━━━━━━━━━━ 164s 24ms/step - loss: 0.2355 - val_loss: 0.2292
Epoch 2/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 145s 23ms/step - loss: 0.2278 - val_loss: 0.2290
Epoch 3/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 145s 23ms/step - loss: 0.2654 - val_loss: 0.2290
Epoch 4/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 144s 23ms/step - loss: 0.2272 - val_loss: 0.2289
Epoch 5/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 144s 23ms/step - loss: 0.2270 - val_loss: 0.2288
Epoch 6/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 147s 24ms/step - loss: 0.2269 - val_loss: 0.2287
Epoch 7/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 147s 24ms/step - loss: 0.2269 - val_loss: 0.2287
Epoch 8/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 148s 24ms/step - loss: 0.2268 - val_loss: 0.2286
Epoch 9/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 148s 24ms/step - loss: 0.2267 - val_loss: 0.2284
Epoch 10/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 148s 24ms/step - loss: 0.2265 - val_loss: 0.2283
Epoch 11/20
6229/6229 ━━━━━━━━━━━━━━━━━━━━ 147s 24ms/step - loss: 0.2265 - val_loss: 0.2285
Epoch 12

In [2]:
# Testing if dataset exists
import os
os.listdir("/kaggle/input/datasets/balraj98/edges2shoes-dataset")

['val', 'metadata.csv', 'train']